# 049 · SAM 2 carries each player's identity (Google Colab, GPU)

**hoop-reid-tracker**, players chapter, experiment 049.

### Why
The online tracker (011b) keeps identities with boxes, motion and appearance, and swaps them when players overlap.
038 then cuts every track at every crossing and gives the pieces to 10 players by appearance. Experiment 046 showed
that fixing the crossings *afterwards* does not help (even perfect links: HOTA 79.7 → 77.9). What is left untested
is changing the **tracker itself**: a segmentation mask follows each player frame to frame (SAM 2 video, as
SAM2MOT and SAMIDARE do), so in a crossing a box belongs to the player whose mask it contains.

### What runs here
1. The online tracker 011b on the 3 held-out **train** clips and the 15 **val** clips: the baseline, and the boxes
   SAM 2 works on (the same boxes, so only the identity mechanism changes).
2. SAM 2 on the 3 train clips with a small grid of settings → the setting is chosen **on train**.
3. That setting **once** on val, against 011b. Optionally 038 + 048 on top of both.

Everything is saved to your Drive as it goes (`MyDrive/hooptrack/runs/049/`): a disconnected session resumes where
it stopped. The scores are computed here; copy the CSVs back to the repository's experiment folder.

### Data, models and licences
| | |
|---|---|
| **Data** | SportsMOT basketball clips (**CC BY-NC 4.0**), downloaded here from Hugging Face with `scripts/download_data.py`. |
| **Your models, from Drive** | the player detector (`Player_detector.pt`, 011) and the court mask (`c06_best.pt`, C06): needed by the online tracker. Optional: the SigLIP ReID (`siglip_reid_sportsmot/`, 028) for 038 on top. They are yours and stay private (AGPL-3.0 + non-commercial data: not redistributed). |
| **SAM 2.1** | `facebook/sam2.1-hiera-small` (Meta, **Apache-2.0**), downloaded from Hugging Face by `transformers`. |
| **Results** | MOT files and scores in your Drive; not published. |

**Before running**: put in `MyDrive/hooptrack/weights/` the files `Player_detector.pt`, `c06_best.pt` (a copy of
`outputs/court_runs/c06_yolo26n_seg/weights/best.pt`) and optionally the folder `siglip_reid_sportsmot/`.
The repository on GitHub must include `scripts/sam2_track.py` (push it first). Runtime: GPU (T4 is enough).
`drive.mount` asks **you** to sign in and mounts **your** Drive.

In [ ]:
# GPU and the repository (cloned from GitHub; pull if already there)
!nvidia-smi --query-gpu=name,memory.total --format=csv
REPO_URL = "https://github.com/VaroLG/hoop-reid-tracker"
!test -d hoop-reid-tracker && git -C hoop-reid-tracker pull -q || git clone -q {REPO_URL}
%cd /content/hoop-reid-tracker
!test -f scripts/sam2_track.py && echo "sam2_track.py found" || echo "MISSING scripts/sam2_track.py: push the repository first"

In [ ]:
# Only what this notebook needs, on top of Colab's own PyTorch (CUDA): never reinstall torch here.
!pip -q install ultralytics==8.4.182 motmetrics==1.4.0 transformers==5.21.0 huggingface_hub==2.2.0 \
    "trackeval @ git+https://github.com/JonathonLuiten/TrackEval.git@12c8791b303e0a0b50f753af204249e622d0281a"
!pip -q install -e . --no-deps
import torch, transformers
print("torch", torch.__version__, "cuda", torch.cuda.is_available(), "| transformers", transformers.__version__)
from transformers import Sam2VideoModel  # noqa: F401  (fails here if this transformers has no SAM 2 video)

## Settings and Drive

In [ ]:
import os, subprocess, json
from pathlib import Path
import pandas as pd

from google.colab import drive
drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive/hooptrack")
WEIGHTS = DRIVE / "weights"
RUNS = DRIVE / "runs" / "049"; RUNS.mkdir(parents=True, exist_ok=True)

DETECTOR = WEIGHTS / "Player_detector.pt"              # 011
COURT_MASK = WEIGHTS / "c06_best.pt"                   # C06
SIGLIP = WEIGHTS / "siglip_reid_sportsmot"             # 028, optional (038 on top)
for p in (DETECTOR, COURT_MASK):
    assert p.exists(), f"missing {p}: copy it to your Drive first"

TRAIN_CLIPS = ["v_-6Os86HzwCs_c009", "v_2j7kLB-vEEk_c010", "v_4LXTUim5anY_c013"]   # held out: values chosen here
MODEL = "facebook/sam2.1-hiera-small"
# (window frames, min share of a mask inside a box, keep bonus = inertia against flicker in piles; 0 = off).
# Runs already on Drive are reused, so the table keeps the earlier settings: w120 s0.4 was the best without inertia.
GRID = [(120, 0.4, k) for k in (0.0, 0.3, 0.6)]
DTYPE = "float16"   # T4: float16 runs on its tensor cores (3x float32; bfloat16 is only emulated). "auto" on A100 / L4

def sh(cmd):
    """Run a command, print its output as it comes (Colab does not show a subprocess's own output), and stop
    the notebook if it fails, with the end of the output in the error."""
    print("$", cmd)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    tail = []
    for line in p.stdout:
        print(line, end="")
        tail = (tail + [line])[-40:]
    assert p.wait() == 0, f"failed: {cmd}\n" + "".join(tail)

## Data
SportsMOT is downloaded split by split and only the basketball clips are kept. If Hugging Face asks for a token,
add yours as a Colab secret named `HF_TOKEN` (key icon on the left).

In [ ]:
try:
    from google.colab import userdata
    os.environ.setdefault("HF_TOKEN", userdata.get("HF_TOKEN"))
except Exception:
    pass
if not Path("data/SportsMOT/dataset/val").is_dir():
    sh("python scripts/download_data.py --splits train val")
print(len(list(Path("data/SportsMOT/dataset/val").iterdir())), "val clips,",
      sum(Path(f"data/SportsMOT/dataset/train/{c}").is_dir() for c in TRAIN_CLIPS), "of 3 held-out train clips")

## 1 · The online tracker (011b): the baseline and the boxes
`configs/player.yaml` is the online tracker of the final pipeline; here it runs on the GPU with your weights.

In [ ]:
def track(split, clips, out):
    if all((out / "results" / f"{c}.txt").is_file() for c in clips):
        print(out, "done"); return
    only = "[" + ",".join(clips) + "]"
    sh(f'python -m hooptrack.cli track --config configs/player.yaml --set data.split={split} "data.only={only}" '
       f"detector.weights={DETECTOR} filters.segmentation.weights={COURT_MASK} "
       f"detector.device=cuda reid.device=cuda filters.segmentation.device=cuda output.dir={out}")

VAL_CLIPS = sorted(p.name for p in Path("data/SportsMOT/dataset/val").iterdir())
track("train", TRAIN_CLIPS, RUNS / "011b_train")
track("val", VAL_CLIPS, RUNS / "011b_val")

## 2 · SAM 2 on the held-out train clips: choose the setting
Each setting writes its own run; `scripts/eval_hota.py` scores them all against the ground truth.

In [ ]:
def sam2(split, run_in, out, window, share, keep=0.0):
    sh(f"python scripts/sam2_track.py --run {run_in} --set data.split={split} --model {MODEL} "
       f"--window {window} --min-share {share} --keep-bonus {keep} --dtype {DTYPE} --out {out}")

def hota(split, runs, csv):
    pairs = " ".join(f"{k}={v}" for k, v in runs.items())
    sh(f"python scripts/eval_hota.py --config configs/player.yaml --set data.split={split} --runs {pairs} --out {csv}")
    t = pd.read_csv(csv)
    return t[t.sequence == "OVERALL"].set_index("run")[["HOTA", "AssA", "IDF1", "MOTA", "IDSW"]]

def tag(w, s, k):
    return f"sam2_w{w}_s{s}" + (f"_k{k}" if k else "")      # k = 0 keeps the names of the runs made before inertia

def done(out, clips):
    return all((out / "results" / f"{c}.txt").exists() for c in clips)

runs, params = {"011b": RUNS / "011b_train"}, {}
for w, s, k in GRID:
    out = RUNS / tag(w, s, k).replace("sam2_", "sam2_train_")
    if not done(out, TRAIN_CLIPS):
        sam2("train", RUNS / "011b_train", out, w, s, k)
    runs[tag(w, s, k)], params[tag(w, s, k)] = out, (w, s, k)
train_table = hota("train", runs, RUNS / "hota_train.csv")
train_table

In [ ]:
# The setting with the best HOTA on train is the one used on val (and nothing is tuned after seeing val)
best = train_table.drop(index="011b")["HOTA"].idxmax()
W_BEST, S_BEST, K_BEST = params[best]
print("chosen on train:", best, "->", W_BEST, S_BEST, K_BEST)
(RUNS / "chosen_on_train.json").write_text(json.dumps({"run": best, "window": W_BEST, "min_share": S_BEST,
                                                       "keep_bonus": K_BEST}))

## 3 · Val, once
The chosen setting on the 15 validation clips, against 011b on the same boxes.

In [ ]:
sam2("val", RUNS / "011b_val", RUNS / "sam2_val", W_BEST, S_BEST, K_BEST)
val_table = hota("val", {"011b": RUNS / "011b_val", "sam2": RUNS / "sam2_val"}, RUNS / "hota_val.csv")
val_table

## 4 · Optional: 038 + 048 on top of both
The final pipeline's identities (exactly 10 players, then the smoothing) on each online run. Needs the SigLIP ReID
in your Drive. The reference on the laptop is 038 + 048: HOTA 79.8, IDF1 89.6 on these 15 clips.

In [ ]:
def final(run_in, out):
    sh(f"python scripts/stitch_tracklets.py --run {run_in} --config configs/player.yaml "
       f"--set reid.type=siglip reid.weights={SIGLIP} reid.device=cuda "
       f"--split-crossings --closed-set 10 --continuity 0.3 --out {out}_038")
    sh(f"python scripts/smooth_body_labels.py --run {out}_038 --out {out}_048")
    return Path(f"{out}_048")

if SIGLIP.is_dir():
    top = {"011b+038+048": final(RUNS / "011b_val", RUNS / "011b_val"),
           "sam2+038+048": final(RUNS / "sam2_val", RUNS / "sam2_val")}
    display(hota("val", top, RUNS / "hota_val_final.csv"))
else:
    print("no SigLIP ReID in", SIGLIP, ": skipped")

## Back on the laptop
Copy `MyDrive/hooptrack/runs/049/hota_*.csv`, `chosen_on_train.json` and the `config.json` of each SAM 2 run to
`experiments/049_sam2_tracker/` and write its README (question → change → results → conclusion), as for every
experiment. Adopt SAM 2 only if it beats the current pipeline on val with the setting chosen on train.